In [ ]:
# q1: Use beam search to predict the best move in a game of chess by evaluating a subset of the most promising moves.  Limit the search depth and beam width for efficiency
# Input: Current board state, beam width, and depth limit. Output: Best move sequence and its evaluation score.

import heapq

graph = {
    'S': [('A', 3), ('B', 6), ('C', 5)],
    'A': [('D', 9), ('E', 8)],
    'B': [('F', 12), ('G', 14)],
    'C': [('H', 7)],
    'H': [('I', 5), ('J', 6)],
    'I': [('K', 1), ('L', 10), ('M', 2)],
    'D': [],'E': [],
    'F': [],'G': [],
    'J': [],'K': [],
    'L': [],'M': []
}

# Beam Search function
def beam_search(start, beam_width=2, depth_limit=3):
    # Initialize the beam with the start state
    beam = [(0, [start])]  # (cumulative cost, path)

    for depth in range(depth_limit):
        
        candidates = []
        # Expand each path in the beam
        
        for cost, path in beam:
            current_node = path[-1]
            for next_node, edge_cost in graph.get(current_node, []):
                new_cost = cost + edge_cost
                new_path = path + [next_node]
                candidates.append((new_cost, new_path))
                
        if not candidates:
            break;  
        
        # Select top-k paths based on the lowest cumulative cost
        beam = heapq.nlargest(beam_width, candidates, key=lambda x: x[0])
        
    best_score, best_path = max(beam, key=lambda x: x[0])
    return best_path, best_score  # Return the best path and its score


# Run Beam Search
current_node = 'S'
depth_limit = 3
beam_width = 2
best_path, cost = beam_search(current_node, beam_width, depth_limit)


# Output
print("Best move sequence:", " → ".join(best_path))
print("Evaluation score:", cost)

Best move sequence: S → B → G
Evaluation score: 20


In [ ]:
# q2: Design a hill climbing algorithm to find the shortest delivery route among a set of locations. 
# Make incremental changes to the route and accept them only if they reduce the total distance.
# Input: List of coordinates for delivery points. Output: Optimized route and the total distance covered.

import math
import random

# Input: coordinates of delivery locations
locations = {
    'A': (2, 3),
    'B': (5, 4),
    'C': (1, 7),
    'D': (6, 8),
    'E': (3, 6)
}

# Function to calculate Euclidean distance
def distance(p1, p2):
    x1, y1 = locations[p1]
    x2, y2 = locations[p2]
    return math.sqrt((x1-x2)**2 + (y1-y2)**2)

# Calculate total route distance
def total_distance(route):
    dist = 0
    for i in range(len(route)-1):
        dist += distance(route[i], route[i+1])
    return dist

# Hill Climbing Algorithm
def hill_climbing(locations):

    # Initial route
    current_route = list(locations.keys())
    random.shuffle(current_route)

    current_distance = total_distance(current_route)

    improved = True

    while improved:
        improved = False

        for i in range(len(current_route)):
            for j in range(i+1, len(current_route)):

                # Swap two locations to create a new route
                new_route = current_route[:]
                new_route[i], new_route[j] = new_route[j], new_route[i]

                new_distance = total_distance(new_route)

                # Accept only if distance decreases
                if new_distance < current_distance:
                    current_route = new_route
                    current_distance = new_distance
                    improved = True

    return current_route, current_distance


# Run Hill Climbing
best_route, best_distance = hill_climbing(locations)

print("Optimized Route:", best_route)
print("Total Distance:", round(best_distance, 2))

Optimized Route: ['C', 'E', 'A', 'B', 'D']
Total Distance: 12.68


In [5]:
# q3: Genetic Algorithm for TSP (10 cities)

import random
import math

# Coordinates of 10 cities
cities = {
    0:(2,3), 1:(5,4), 2:(1,7), 3:(6,8), 4:(3,6),
    5:(7,2), 6:(4,9), 7:(8,5), 8:(9,3), 9:(2,8)
}

# Distance between two cities
def distance(c1, c2):
    x1,y1 = cities[c1]
    x2,y2 = cities[c2]
    return math.sqrt((x1-x2)**2 + (y1-y2)**2)

# Fitness = total route distance
def fitness(route):
    total = 0
    for i in range(len(route)-1):
        total += distance(route[i], route[i+1])
    total += distance(route[-1], route[0])  # return to start
    return total

# Selection (Roulette wheel using inverse distance)
def roulette_wheel_selection(population):
    fitness_values = [1/fitness(route) for route in population]
    total = sum(fitness_values)
    probs = [f/total for f in fitness_values]
    parents = random.choices(population, probs, k=2)
    return parents

# Crossover (Order crossover style)
def crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))

    child = [-1]*size
    child[start:end] = parent1[start:end]

    p2_index = 0
    for i in range(size):
        if child[i] == -1:
            while parent2[p2_index] in child:
                p2_index += 1
            child[i] = parent2[p2_index]

    return child

# Mutation (swap two cities)
def mutate(route, mutation_rate):
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]
    return route

# Genetic Algorithm
def genetic_algorithm(pop_size, generations, mutation_rate):

    # Initial population (random routes)
    population = [random.sample(range(10), 10) for _ in range(pop_size)]

    for _ in range(generations):

        new_population = []

        for _ in range(pop_size):
            parent1, parent2 = roulette_wheel_selection(population)

            child = crossover(parent1, parent2)

            child = mutate(child, mutation_rate)

            new_population.append(child)

        population = new_population

    # Best route (minimum distance)
    best_route = min(population, key=fitness)
    return best_route, fitness(best_route)


# Parameters
population_size = 20
generations = 50
mutation_rate = 0.05

# Run GA
best_route, best_distance = genetic_algorithm(population_size, generations, mutation_rate)

print("Best Route:", best_route)
print("Total Distance:", round(best_distance,2))

Best Route: [0, 1, 9, 2, 4, 7, 8, 5, 6, 3]
Total Distance: 37.64


In [6]:
# q4: Design a Beam Search algorithm to allocate job tasks to available processors in a distributed system. 
# The algorithm should consider factors such as execution time,processor load, and priority.
# Input: A list of job tasks with execution times and priorities, and a set of available processors.
# Output: Optimized task allocation that minimizes the maximum load on any processor.

import heapq

# Job tasks: (TaskName, ExecutionTime, Priority)
tasks = [
    ("T1", 4, 3),
    ("T2", 2, 2),
    ("T3", 6, 1),
    ("T4", 3, 2),
    ("T5", 5, 1)
]

# Available processors
processors = ["P1", "P2", "P3"]

beam_width = 2


# Evaluation function (minimize max processor load)
def evaluate(loads):
    return max(loads.values())


def beam_search_task_allocation(tasks, processors, beam_width):

    # Initial state
    initial_load = {p:0 for p in processors}
    beam = [(0, {}, initial_load)]  
    # (score, allocation, processor_loads)

    for task, exec_time, priority in tasks:

        candidates = []

        for score, allocation, loads in beam:

            for p in processors:

                new_alloc = allocation.copy()
                new_loads = loads.copy()

                # Assign task to processor
                new_alloc[task] = p
                new_loads[p] += exec_time

                # Score considers processor load and priority
                new_score = evaluate(new_loads) - priority

                candidates.append((new_score, new_alloc, new_loads))

        # Keep best k allocations
        beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])

    best = min(beam, key=lambda x: x[0])
    return best


score, allocation, loads = beam_search_task_allocation(tasks, processors, beam_width)

print("Task Allocation:")
for t, p in allocation.items():
    print(t, "->", p)

print("\nProcessor Loads:", loads)
print("Maximum Load:", max(loads.values()))

Task Allocation:
T1 -> P1
T2 -> P2
T3 -> P3
T4 -> P2
T5 -> P1

Processor Loads: {'P1': 9, 'P2': 5, 'P3': 6}
Maximum Load: 9
